In [1]:
!git clone --depth 1 https://github.com/RangiLyu/nanodet.git /kaggle/working/nanodet

with open('/kaggle/working/nanodet/nanodet/data/collate.py', 'r') as f:
    content = f.read()
content = content.replace(
    'from torch._six import string_classes',
    'string_classes = (str, bytes)'
)
with open('/kaggle/working/nanodet/nanodet/data/collate.py', 'w') as f:
    f.write(content)

with open('/kaggle/working/nanodet/requirements.txt', 'r') as f:
    reqs = [line for line in f if not line.strip().lower().startswith(('torch', 'torchvision'))]
with open('/kaggle/working/nanodet/requirements.txt', 'w') as f:
    f.writelines(reqs)

!pip install -q -r /kaggle/working/nanodet/requirements.txt

import sys
sys.path.insert(0, '/kaggle/working/nanodet')

import nanodet
print("nanodet imported OK")

Cloning into '/kaggle/working/nanodet'...
remote: Enumerating objects: 327, done.
remote: Counting objects: 100% (327/327), done.
remote: Compressing objects: 100% (290/290), done.
remote: Total 327 (delta 49), reused 174 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (327/327), 1.42 MiB | 7.59 MiB/s, done.
Resolving deltas: 100% (49/49), done.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.5/829.5 kB 10.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 51.5 MB/s eta 0:00:00a 0:00:01
nanodet imported OK


In [2]:
import torch
import torch.nn as nn
from nanodet.util import cfg, load_config
from nanodet.model.arch import build_model

MODEL_CONFIG_YAML = """
model:
  arch:
    name: NanoDetPlus
    detach_epoch: 10
    backbone:
      name: ShuffleNetV2
      model_size: 1.0x
      out_stages: [2,3,4]
      activation: LeakyReLU
      pretrain: False
    fpn:
      name: GhostPAN
      in_channels: [116, 232, 464]
      out_channels: 96
      kernel_size: 5
      num_extra_level: 1
      use_depthwise: True
      activation: LeakyReLU
    head:
      name: NanoDetPlusHead
      num_classes: 3
      input_channel: 96
      feat_channels: 96
      stacked_convs: 2
      kernel_size: 5
      strides: [8, 16, 32, 64]
      activation: LeakyReLU
      reg_max: 7
      norm_cfg:
        type: BN
      loss:
        loss_qfl:
          name: QualityFocalLoss
          use_sigmoid: True
          beta: 2.0
          loss_weight: 1.0
        loss_dfl:
          name: DistributionFocalLoss
          loss_weight: 0.25
        loss_bbox:
          name: GIoULoss
          loss_weight: 2.0
    aux_head:
      name: SimpleConvHead
      num_classes: 3
      input_channel: 192
      feat_channels: 192
      stacked_convs: 4
      strides: [8, 16, 32, 64]
      activation: LeakyReLU
      reg_max: 7
"""
with open('/kaggle/working/model_only_config.yml', 'w') as f:
    f.write(MODEL_CONFIG_YAML)

load_config(cfg, '/kaggle/working/model_only_config.yml')
base_model = build_model(cfg.model)

CONFIG_A_CHECKPOINT = "/kaggle/input/datasets/rainbowpotatoes/flir-aligned-nanodet/kaggle_upload/checkpoint/model_best.ckpt"
ckpt = torch.load(CONFIG_A_CHECKPOINT, map_location="cpu")
avg_model_weights = {
    k.replace("avg_model.", "", 1): v
    for k, v in ckpt["state_dict"].items()
    if k.startswith("avg_model.")
}
base_model.load_state_dict(avg_model_weights, strict=True)
print("Loaded avg_model weights from Config A checkpoint (strict=True, no mismatches)")

base_model.eval()
for p in base_model.parameters():
    p.requires_grad = False


# stage 0 branch: 3 convs, 416->52 (matches Config B's design exactly)
class ThermalBranchStage0(nn.Module):
    def __init__(self, out_channels, size="small"):
        super().__init__()
        mid = 16 if size == "small" else 32
        self.net = nn.Sequential(
            nn.Conv2d(1, mid, 3, 2, 1), nn.BatchNorm2d(mid), nn.ReLU(inplace=True),
            nn.Conv2d(mid, mid*2, 3, 2, 1), nn.BatchNorm2d(mid*2), nn.ReLU(inplace=True),
            nn.Conv2d(mid*2, out_channels, 3, 2, 1), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
        )
    def forward(self, thermal):
        return self.net(thermal)


# stage 1 branch: 4 convs, 416->26 (one more downsample, same philosophy)
class ThermalBranchStage1(nn.Module):
    def __init__(self, out_channels, size="small"):
        super().__init__()
        mid = 16 if size == "small" else 32
        self.net = nn.Sequential(
            nn.Conv2d(1, mid, 3, 2, 1), nn.BatchNorm2d(mid), nn.ReLU(inplace=True),
            nn.Conv2d(mid, mid*2, 3, 2, 1), nn.BatchNorm2d(mid*2), nn.ReLU(inplace=True),
            nn.Conv2d(mid*2, mid*2, 3, 2, 1), nn.BatchNorm2d(mid*2), nn.ReLU(inplace=True),
            nn.Conv2d(mid*2, out_channels, 3, 2, 1), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
        )
    def forward(self, thermal):
        return self.net(thermal)


class ChannelAttentionFusion(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.gate = nn.Sequential(
            nn.Linear(channels*2, channels//reduction), nn.ReLU(inplace=True),
            nn.Linear(channels//reduction, channels*2), nn.Sigmoid(),
        )
    def forward(self, rgb_feat, thermal_feat):
        b, c, h, w = rgb_feat.shape
        pooled = torch.cat([self.pool(rgb_feat).view(b,c), self.pool(thermal_feat).view(b,c)], dim=1)
        weights = self.gate(pooled)
        w_rgb, w_thermal = weights[:, :c].view(b,c,1,1), weights[:, c:].view(b,c,1,1)
        return rgb_feat * w_rgb + thermal_feat * w_thermal


thermal_branch_0 = ThermalBranchStage0(out_channels=116, size="small")
fusion_module_0 = ChannelAttentionFusion(channels=116)
thermal_branch_1 = ThermalBranchStage1(out_channels=232, size="small")
fusion_module_1 = ChannelAttentionFusion(channels=232)

_current_thermal = {}

def fuse_hook(module, inputs, output):
    feat0, feat1, feat2 = output
    t_feat0 = thermal_branch_0(_current_thermal["x"])
    feat0 = fusion_module_0(feat0, t_feat0)
    t_feat1 = thermal_branch_1(_current_thermal["x"])
    feat1 = fusion_module_1(feat1, t_feat1)
    return (feat0, feat1, feat2)

base_model.backbone.register_forward_hook(fuse_hook)

trainable = (sum(p.numel() for p in thermal_branch_0.parameters()) +
             sum(p.numel() for p in fusion_module_0.parameters()) +
             sum(p.numel() for p in thermal_branch_1.parameters()) +
             sum(p.numel() for p in fusion_module_1.parameters()))
frozen = sum(p.numel() for p in base_model.parameters())
print(f"trainable params: {trainable:,}  frozen params: {frozen:,}")

/usr/local/lib/python3.12/dist-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)


model size is  1.0x
init weights...
Finish initialize NanoDet-Plus Head.
Loaded avg_model weights from Config A checkpoint (strict=True, no mismatches)
trainable params: 154,519  frozen params: 4,167,895


In [3]:
import cv2
import json
import numpy as np
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

DATASET_ROOT = Path("/kaggle/input/datasets/rainbowpotatoes/flir-aligned-nanodet/kaggle_upload/flir_align")
IMG_SIZE = 416

NORM_MEAN = np.array([103.53, 116.28, 123.675], dtype=np.float32).reshape(1, 1, 3)
NORM_STD = np.array([57.375, 57.12, 58.395], dtype=np.float32).reshape(1, 1, 3)


class PairedFlirDatasetNanoDet(Dataset):
    def __init__(self, split: str):
        assert split in ("train", "val")
        img_subdir = "train" if split == "train" else "test"
        ann_name = "train_new_intid.json" if split == "train" else "test_new_intid.json"

        self.rgb_dir = DATASET_ROOT / "visible" / img_subdir
        self.thermal_dir = DATASET_ROOT / "thermal" / img_subdir

        with open(DATASET_ROOT / "coco_annotations" / ann_name) as f:
            data = json.load(f)

        self.images_by_id = {img["id"]: img for img in data["images"]}
        self.anns_by_image = {}
        for ann in data["annotations"]:
            self.anns_by_image.setdefault(ann["image_id"], []).append(ann)

        self.image_ids = sorted(self.images_by_id.keys())

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_info = self.images_by_id[img_id]
        file_name = img_info["file_name"]
        orig_w, orig_h = img_info["width"], img_info["height"]

        rgb = cv2.imread(str(self.rgb_dir / file_name))
        rgb = cv2.resize(rgb, (IMG_SIZE, IMG_SIZE))
        rgb = rgb.astype(np.float32)
        rgb = (rgb - NORM_MEAN) / NORM_STD
        rgb = rgb.transpose(2, 0, 1)

        thermal = cv2.imread(str(self.thermal_dir / file_name), cv2.IMREAD_GRAYSCALE)
        thermal = cv2.resize(thermal, (IMG_SIZE, IMG_SIZE))
        thermal = thermal.astype(np.float32) / 255.0
        thermal = thermal[np.newaxis, :, :]

        scale_x = IMG_SIZE / orig_w
        scale_y = IMG_SIZE / orig_h
        anns = self.anns_by_image.get(img_id, [])
        boxes, labels = [], []
        for ann in anns:
            x, y, w, h = ann["bbox"]
            x1, y1, x2, y2 = x * scale_x, y * scale_y, (x + w) * scale_x, (y + h) * scale_y
            boxes.append([x1, y1, x2, y2])
            labels.append(ann["category_id"])
        boxes = np.array(boxes, dtype=np.float32) if boxes else np.zeros((0, 4), dtype=np.float32)
        labels = np.array(labels, dtype=np.int64) if labels else np.zeros((0,), dtype=np.int64)

        return {"rgb": rgb, "thermal": thermal, "boxes": boxes, "labels": labels,
                "img_id": img_id, "orig_w": orig_w, "orig_h": orig_h}


def collate_fn(batch):
    rgb_batch = torch.stack([torch.from_numpy(item["rgb"]) for item in batch])
    thermal_batch = torch.stack([torch.from_numpy(item["thermal"]) for item in batch])
    gt_bboxes = [item["boxes"] for item in batch]
    gt_labels = [item["labels"] for item in batch]
    ids = [item["img_id"] for item in batch]
    heights = torch.tensor([IMG_SIZE] * len(batch))
    widths = torch.tensor([IMG_SIZE] * len(batch))
    gt_meta = {
        "img": rgb_batch, "gt_bboxes": gt_bboxes, "gt_labels": gt_labels,
        "gt_bboxes_ignore": None,
        "img_info": {"height": heights, "width": widths, "id": torch.tensor(ids)},
        "warp_matrix": [np.eye(3) for _ in range(len(batch))],
    }
    return gt_meta, thermal_batch


def eval_collate_fn(batch):
    rgb_batch = torch.stack([torch.from_numpy(item["rgb"]) for item in batch])
    thermal_batch = torch.stack([torch.from_numpy(item["thermal"]) for item in batch])
    gt_bboxes = [item["boxes"] for item in batch]
    gt_labels = [item["labels"] for item in batch]
    ids = [item["img_id"] for item in batch]
    orig_heights = torch.tensor([item["orig_h"] for item in batch])
    orig_widths = torch.tensor([item["orig_w"] for item in batch])
    warp_matrices = []
    for item in batch:
        scale_x = IMG_SIZE / item["orig_w"]
        scale_y = IMG_SIZE / item["orig_h"]
        warp_matrices.append(np.array([[scale_x, 0, 0], [0, scale_y, 0], [0, 0, 1]], dtype=np.float64))
    gt_meta = {
        "img": rgb_batch, "gt_bboxes": gt_bboxes, "gt_labels": gt_labels,
        "gt_bboxes_ignore": None,
        "img_info": {"height": orig_heights, "width": orig_widths, "id": torch.tensor(ids)},
        "warp_matrix": warp_matrices,
    }
    return gt_meta, thermal_batch


train_dataset = PairedFlirDatasetNanoDet("train")
val_dataset = PairedFlirDatasetNanoDet("val")

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn, num_workers=2)
eval_val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=eval_collate_fn, num_workers=2)

print(f"train: {len(train_dataset)} images, val: {len(val_dataset)} images")

train: 4129 images, val: 1013 images


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base_model.to(device)
thermal_branch_0.to(device)
fusion_module_0.to(device)
thermal_branch_1.to(device)
fusion_module_1.to(device)
print(f"Using device: {device}")

Using device: cuda


In [5]:
EPOCHS = 30
CHECKPOINT_PATH = "/kaggle/working/config_c_nanodet_multi_fusion_best.pt"

optimizer = torch.optim.AdamW(
    list(thermal_branch_0.parameters()) + list(fusion_module_0.parameters()) +
    list(thermal_branch_1.parameters()) + list(fusion_module_1.parameters()),
    lr=1e-3,
)

best_val_loss = float("inf")
n_train_batches = len(train_loader)

for epoch in range(EPOCHS):
    base_model.set_epoch(epoch)
    thermal_branch_0.train(); fusion_module_0.train()
    thermal_branch_1.train(); fusion_module_1.train()

    train_loss_total = 0.0
    for batch_num, (gt_meta, thermal_batch) in enumerate(train_loader):
        gt_meta["img"] = gt_meta["img"].to(device)
        thermal_batch = thermal_batch.to(device)
        _current_thermal["x"] = thermal_batch

        optimizer.zero_grad()
        head_out, loss, loss_states = base_model.forward_train(gt_meta)
        loss.backward()
        optimizer.step()

        train_loss_total += loss.item()

        if batch_num % 20 == 0:
            print(f"  epoch {epoch+1}, batch {batch_num}/{n_train_batches}, running loss: {train_loss_total/(batch_num+1):.4f}")

    avg_train_loss = train_loss_total / len(train_loader)

    thermal_branch_0.eval(); fusion_module_0.eval()
    thermal_branch_1.eval(); fusion_module_1.eval()

    val_loss_total = 0.0
    with torch.no_grad():
        for gt_meta, thermal_batch in val_loader:
            gt_meta["img"] = gt_meta["img"].to(device)
            thermal_batch = thermal_batch.to(device)
            _current_thermal["x"] = thermal_batch
            head_out, loss, loss_states = base_model.forward_train(gt_meta)
            val_loss_total += loss.item()

    avg_val_loss = val_loss_total / len(val_loader)
    print(f"epoch {epoch+1}/{EPOCHS}  train_loss={avg_train_loss:.4f}  val_loss={avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save({
            "thermal_branch_0": thermal_branch_0.state_dict(),
            "fusion_module_0": fusion_module_0.state_dict(),
            "thermal_branch_1": thermal_branch_1.state_dict(),
            "fusion_module_1": fusion_module_1.state_dict(),
        }, CHECKPOINT_PATH)
        print(f"  saved new best checkpoint (val_loss={avg_val_loss:.4f})")

print(f"Training complete. Best val loss: {best_val_loss:.4f}")

/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


  epoch 1, batch 0/130, running loss: 5.6444
  epoch 1, batch 20/130, running loss: 3.2662
  epoch 1, batch 40/130, running loss: 2.6871
  epoch 1, batch 60/130, running loss: 2.3616
  epoch 1, batch 80/130, running loss: 2.1631
  epoch 1, batch 100/130, running loss: 2.0268
  epoch 1, batch 120/130, running loss: 1.9346
epoch 1/30  train_loss=1.8990  val_loss=1.7494
  saved new best checkpoint (val_loss=1.7494)
  epoch 2, batch 0/130, running loss: 1.4734
  epoch 2, batch 20/130, running loss: 1.3994
  epoch 2, batch 40/130, running loss: 1.4232
  epoch 2, batch 60/130, running loss: 1.4043
  epoch 2, batch 80/130, running loss: 1.4027
  epoch 2, batch 100/130, running loss: 1.3962
  epoch 2, batch 120/130, running loss: 1.3897
epoch 2/30  train_loss=1.3949  val_loss=1.7005
  saved new best checkpoint (val_loss=1.7005)
  epoch 3, batch 0/130, running loss: 1.3339
  epoch 3, batch 20/130, running loss: 1.3707
  epoch 3, batch 40/130, running loss: 1.3662
  epoch 3, batch 60/130, runnin

In [6]:
from pycocotools.coco import COCO
from nanodet.evaluator import CocoDetectionEvaluator

val_dataset.coco_api = COCO(DATASET_ROOT / "coco_annotations" / "test_new_intid.json")
val_dataset.class_names = ["car", "person", "bicycle"]
val_dataset.cat_ids = sorted(val_dataset.coco_api.getCatIds())

evaluator = CocoDetectionEvaluator(val_dataset)

thermal_branch_0.eval(); fusion_module_0.eval()
thermal_branch_1.eval(); fusion_module_1.eval()

all_results = {}
with torch.no_grad():
    for gt_meta, thermal_batch in eval_val_loader:
        gt_meta["img"] = gt_meta["img"].to(device)
        thermal_batch = thermal_batch.to(device)
        _current_thermal["x"] = thermal_batch

        preds = base_model(gt_meta["img"])
        det_results = base_model.head.post_process(preds, gt_meta)
        all_results.update(det_results)

print("=== Config C, full val set ===")
eval_results = evaluator.evaluate(all_results, save_dir="/kaggle/working/")
print(eval_results)

loading annotations into memory...
Done (t=0.04s)
creating index...
index created!
=== Config C, full val set ===
Loading and preparing results...
DONE (t=0.87s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=8.61s).
Accumulating evaluation results...
DONE (t=0.95s).
{'mAP': np.float64(0.26876569427703734), 'AP_50': np.float64(0.5684458631551123), 'AP_75': np.float64(0.2114813218067967), 'AP_small': np.float64(0.13243689145912965), 'AP_m': np.float64(0.34125557281121205), 'AP_l': np.float64(0.4143412469231659)}


In [7]:
import os
CHECKPOINT_PATH = "/kaggle/working/config_c_nanodet_multi_fusion_best.pt"
print("Checkpoint exists on disk:", os.path.exists(CHECKPOINT_PATH))

ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu")
current_sd = thermal_branch_0.state_dict()
saved_sd = ckpt["thermal_branch_0"]

match = all(torch.equal(current_sd[k].cpu(), saved_sd[k]) for k in saved_sd)
print("In-memory thermal_branch_0 matches the saved trained checkpoint:", match)

Checkpoint exists on disk: True
In-memory thermal_branch_0 matches the saved trained checkpoint: False
